In [0]:

from pyspark.sql import functions as F

CATALOG = "marketplace_catalog"
SCHEMA = "team6_marketplace"

def read_table(name):
    return spark.table(f"{CATALOG}.{SCHEMA}.{name}")

def save_gold(df, name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{SCHEMA}.{name}")
    )

print("Gold layer configuration completed.")


Gold layer configuration completed.


In [0]:

fact = read_table("fact_order_items")
products = read_table("dim_product")
orders = read_table("silver_orders")
translations = read_table("bronze_category_translation")

# dim_product only has the Portuguese category name; the English name
# comes from the category translation table.
product_categories = (
    products
    .select("product_id", "product_category_name")
    .join(
        translations.select("product_category_name",
                            "product_category_name_english"),
        "product_category_name", "left"
    )
    .select("product_id", "product_category_name_english")
)

monthly_category_revenue = (
    fact
    .join(product_categories, "product_id", "left")
    .join(
        orders.select("order_id", "order_purchase_timestamp"),
        "order_id", "left"
    )
    .withColumn(
        "year_month",
        F.date_format("order_purchase_timestamp", "yyyy-MM")
    )
    .groupBy("year_month", "product_category_name_english")
    .agg(
        F.sum("price").alias("total_revenue"),
        F.sum("freight_value").alias("total_freight"),
        F.countDistinct("order_id").alias("total_orders")
    )
    .orderBy("year_month", F.desc("total_revenue"))
)

save_gold(monthly_category_revenue, "gold_monthly_category_revenue")

display(monthly_category_revenue)


year_month,product_category_name_english,total_revenue,total_freight,total_orders
2016-09,health_beauty,134.97,8.49,1
2016-09,furniture_decor,72.89,63.34,1
2016-09,telephony,59.5,15.56,1
2016-10,furniture_decor,5807.889999999999,1244.3899999999999,51
2016-10,perfumery,5688.7,535.14,30
2016-10,health_beauty,4552.51,940.87,44
2016-10,toys,4465.09,520.99,25
2016-10,consoles_games,3882.26,207.16,10
2016-10,watches_gifts,3360.24,107.8,5
2016-10,sports_leisure,3333.6400000000003,593.47,19


In [0]:

customers = read_table("dim_customer")

revenue_by_state = (
    fact
    .join(customers.select("customer_id", "customer_state"),
          "customer_id", "left")
    .groupBy("customer_state")
    .agg(
        F.sum("price").alias("total_revenue"),
        F.countDistinct("order_id").alias("total_orders"),
        F.count("*").alias("total_order_items")
    )
    .orderBy(F.desc("total_revenue"))
)

save_gold(revenue_by_state, "gold_revenue_by_state")

display(revenue_by_state)


customer_state,total_revenue,total_orders,total_order_items
SP,5202955.049999564,41375,47449
RJ,1824092.6700000311,12762,14579
MG,1585308.0300000198,11544,13129
RS,750304.0199999912,5432,6235
PR,683083.7599999921,4998,5740
SC,520553.339999996,3612,4176
BA,511349.9899999971,3358,3799
DF,302603.9399999997,2125,2406
GO,294591.9499999999,2007,2333
ES,275037.3099999998,2025,2256


In [0]:

orders = read_table("silver_orders")
customers = read_table("dim_customer")

delivery_by_state = (
    orders
    .join(
        customers.select("customer_id", "customer_state"),
        "customer_id",
        "left"
    )
    .filter(F.col("order_delivered_customer_date").isNotNull())
    .withColumn(
        "delivery_delay_days",
        F.datediff(
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        )
    )
    .groupBy("customer_state")
    .agg(
        F.count("order_id").alias("delivered_orders"),
        F.round(F.avg("delivery_delay_days"), 2).alias("avg_delay_days"),
        F.sum(
            F.when(F.col("delivery_delay_days") > 0, 1).otherwise(0)
        ).alias("late_orders")
    )
    .orderBy(F.desc("avg_delay_days"))
)

save_gold(delivery_by_state, "gold_delivery_by_state")

display(delivery_by_state)


customer_state,delivered_orders,avg_delay_days,late_orders
AL,397,-8.71,85
MA,717,-9.57,125
SE,335,-10.02,51
ES,1995,-10.5,214
BA,3256,-10.79,396
CE,1279,-10.8,176
MS,701,-11.05,68
SP,40495,-11.08,1820
PI,476,-11.31,66
SC,3547,-11.51,291


In [0]:

sellers = read_table("dim_seller")
orders = read_table("silver_orders")

seller_scorecard = (
    fact
    .join(
        sellers.select("seller_id", "seller_city", "seller_state"),
        "seller_id", "left"
    )
    .join(
        orders.select(
            "order_id",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ).alias("o"),
        "order_id", "left"
    )
    .withColumn(
        "delivery_delay_days",
        F.datediff(
            F.col("o.order_delivered_customer_date"),
            F.col("o.order_estimated_delivery_date")
        )
    )
    .groupBy("seller_id", "seller_city", "seller_state")
    .agg(
        F.round(F.sum("price"), 2).alias("seller_revenue"),
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.avg("review_score"), 2).alias("avg_review_score"),
        F.round(F.avg("delivery_delay_days"), 2).alias("avg_delivery_delay")
    )
    .orderBy(F.desc("seller_revenue"))
)

save_gold(seller_scorecard, "gold_seller_scorecard")

display(seller_scorecard)


seller_id,seller_city,seller_state,seller_revenue,total_orders,avg_review_score,avg_delivery_delay
4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,229472.63,1132,4.12,-11.16
53243585a1d6dc2643021fd1853d8905,lauro de freitas,BA,222776.05,358,4.09,-11.61
4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,200472.92,1806,3.81,-9.64
fa1c13f2614d7b5c4749cbc52fecda94,sumare,SP,194042.03,585,4.34,-10.31
7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,187923.89,982,3.34,-11.53
7e93a43ef30c4f03f38b393420bc753a,barueri,SP,176431.87,336,4.21,-12.16
da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,160236.57,1314,4.07,-11.24
7a67c85e85bb2ce8582c35f2203ad736,sao paulo,SP,141745.53,1160,4.23,-11.84
1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,138968.55,915,3.86,-11.1
955fee9216a65b617aa5c0531780ce60,sao paulo,SP,135171.7,1287,4.05,-10.33


In [0]:

late_delivery_analysis = (
    fact
    .withColumn(
        "delivery_delay_days",
        F.datediff(
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        )
    )
    .filter(
        F.col("delivery_delay_days").isNotNull() &
        F.col("review_score").isNotNull()
    )
    .withColumn(
        "delivery_category",
        F.when(F.col("delivery_delay_days") > 0, "Late")
         .when(F.col("delivery_delay_days") == 0, "On Time")
         .otherwise("Early")
    )
    .groupBy("delivery_category")
    .agg(
        F.count("*").alias("order_items"),
        F.round(F.avg("review_score"), 2).alias("average_review_score")
    )
)

save_gold(late_delivery_analysis, "gold_late_delivery_review_analysis")

display(late_delivery_analysis)


delivery_category,order_items,average_review_score
Early,100305,4.21
On Time,1423,3.99
Late,7033,2.26


In [0]:

gold_tables = [
    "gold_monthly_category_revenue",
    "gold_revenue_by_state",
    "gold_delivery_by_state",
    "gold_seller_scorecard",
    "gold_late_delivery_review_analysis"
]

for name in gold_tables:
    df = read_table(name)
    print(name, "| Rows:", df.count(), "| Columns:", len(df.columns))


gold_monthly_category_revenue | Rows: 1274 | Columns: 5
gold_revenue_by_state | Rows: 27 | Columns: 4
gold_delivery_by_state | Rows: 27 | Columns: 4
gold_seller_scorecard | Rows: 3095 | Columns: 7
gold_late_delivery_review_analysis | Rows: 3 | Columns: 3
